In [1]:
import pandas as pd
import numpy as np  

In [2]:
df = pd.read_csv('Finance.csv')
df = df.sort_values(by='Date', ascending=True)
print(df.head())

       Symbol        Date     Open     High      Low    Close Percent Change  \
1157  FINANCE  2021-03-31  1169.34  1182.57  1166.94  1176.73         0.00 %   
1156  FINANCE  2021-04-01  1177.74  1199.50  1175.39  1199.50         0.00 %   
1155  FINANCE  2021-04-04  1206.31  1243.46  1206.31  1243.46         0.00 %   
1154  FINANCE  2021-04-05  1258.95  1259.93  1227.41  1242.51         0.00 %   
1153  FINANCE  2021-04-06  1235.81  1239.20  1225.35  1239.15         0.00 %   

     Volume Turn Over  
1157      -         -  
1156      -         -  
1155      -         -  
1154      -         -  
1153      -         -  


In [3]:
df['tp'] = (df['Close']+df['High']+df['Low'])/3

In [4]:
# Clean thousands separators before numeric conversion
df['Volume'] = pd.to_numeric(df['Volume'].astype(str).str.replace(',', '', regex=False), errors='coerce')
df['rmf'] = df['tp'] * df['Volume']

In [5]:
#Find postive and negative money flow
df['tp_diff'] = df['tp'].diff()
df['Postive_tp']= df['tp_diff'] > 0
df['Negative_tp']= df['tp_diff'] < 0


In [6]:
df['pos_flow'] = df['rmf'].where(df['Postive_tp'], 0)
df['neg_flow'] = df['rmf'].where(df['Negative_tp'], 0)

pos_sum = df['pos_flow'].rolling(14).sum()
neg_sum = df['neg_flow'].rolling(14).sum()

mfr = pos_sum / neg_sum
df['mfi'] = 100 - (100 / (1 + mfr))

In [7]:
df.tail()

,Symbol,Date,Open,High,Low,Close,Percent Change,Volume,Turn Over,tp,rmf,tp_diff,Postive_tp,Negative_tp,pos_flow,neg_flow,mfi
4,FINANCE,2026-03-24,2688.65,2705.17,2666.56,2705.17,0.51 %,531812206.1,-,2692.300000,1.431798e+12,8.030000,True,False,1.431798e+12,0.000000e+00,74.697818
3,FINANCE,2026-03-25,2701.96,2718.97,2641.28,2648.40,-2.10 %,415780210.6,-,2669.550000,1.109946e+12,-22.750000,False,True,0.000000e+00,1.109946e+12,68.874805
2,FINANCE,2026-03-26,2657.62,2657.62,2627.47,2652.93,0.17 %,220500026.2,-,2646.006667,5.834445e+11,-23.543333,False,True,0.000000e+00,5.834445e+11,64.079611
1,FINANCE,2026-03-29,2654.21,2655.04,2550.04,2563.43,-3.37 %,329912492.3,-,2589.503333,8.543095e+11,-56.503333,False,True,0.000000e+00,8.543095e+11,58.547407
0,FINANCE,2026-03-30,2561.86,2576.27,2499.66,2512.15,-2.00 %,225810191.0,-,2529.360000,5.711553e+11,-60.143333,False,True,0.000000e+00,5.711553e+11,56.400857


In [8]:
# Prepare data for plotting: convert date to datetime
df_plot = df.copy()
df_plot['Date'] = pd.to_datetime(df_plot['Date'])

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots with 2 rows and shared x-axis
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    row_heights=[0.7, 0.3],
    subplot_titles=("Candlestick Chart", "Money Flow Index (MFI)")
)

# Convert date to string to use as categorical axis (preserves gaps)
df_plot['Date_str'] = df_plot['Date'].astype(str)

# Add candlestick chart to the first subplot
fig.add_trace(
    go.Candlestick(
        x=df_plot['Date_str'],
        open=df_plot['Open'],
        high=df_plot['High'],
        low=df_plot['Low'],
        close=df_plot['Close'],
        name='OHLC',
        increasing_line_color='green',
        decreasing_line_color='red',
    ),
    row=1, col=1
)

# Add MFI line chart to the second subplot
fig.add_trace(
    go.Scatter(
        x=df_plot['Date_str'],
        y=df_plot['mfi'],
        mode='lines',
        name='MFI',
        line=dict(color='blue', width=2),
        fill='tozeroy',
        fillcolor='rgba(0, 0, 255, 0.2)'
    ),
    row=2, col=1
)

# Add horizontal line at 70 and 30 for overbought/oversold
fig.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="Overbought", row=2)
fig.add_hline(y=30, line_dash="dash", line_color="green", annotation_text="Oversold", row=2)

# Update layout
fig.update_layout(
    height=700,
    title_text="Trading View with MFI Indicator",
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    template='plotly_white'
)

# Update y-axes labels
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="MFI", row=2, col=1)

fig.show()

In [10]:
# Swing detection function
def find_swings(series, lookback=10):
    highs, lows = [], []
    for i in range(lookback, len(series) - lookback):
        window = series.iloc[i-lookback: lookback + i +1]
        current = float(series.iloc[i])
        if current == float(window.max()):
            highs.append(i)
        if current == float(window.min()):
            lows.append(i)
    return highs, lows

# Find swing highs and lows for price
high_idx, low_idx = find_swings(df_plot["Close"])

# Divergence detection
regular_bullish, regular_bearish = [], []
hidden_bullish, hidden_bearish = [], []

# Regular Bullish: Price LL, MFI HL
for i in range(1, len(low_idx)):
    p1, p2 = low_idx[i - 1], low_idx[i]
    if df_plot["Close"].iloc[p2] < df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] > df_plot["mfi"].iloc[p1]:
        regular_bullish.append((p1, p2))

# Regular Bearish: Price HH, MFI LH
for i in range(1, len(high_idx)):
    p1, p2 = high_idx[i - 1], high_idx[i]
    if df_plot["Close"].iloc[p2] > df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] < df_plot["mfi"].iloc[p1]:
        regular_bearish.append((p1, p2))

# Hidden Bullish: Price HL, MFI LL
for i in range(1, len(low_idx)):
    p1, p2 = low_idx[i-1], low_idx[i]
    if df_plot["Close"].iloc[p2] > df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] < df_plot["mfi"].iloc[p1]:
        hidden_bullish.append((p1, p2))

# Hidden Bearish: Price LH, MFI HH
for i in range(1, len(high_idx)):
    p1, p2 = high_idx[i-1], high_idx[i]
    if df_plot["Close"].iloc[p2] < df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] > df_plot["mfi"].iloc[p1]:
        hidden_bearish.append((p1, p2))

print("Regular Bullish Divergences:", regular_bullish)
print("Regular Bearish Divergences:", regular_bearish)
print("Hidden Bullish Divergences:", hidden_bullish)
print("Hidden Bearish Divergences:", hidden_bearish)

Regular Bullish Divergences: [(260, 294), (460, 483), (483, 506), (585, 623), (679, 691), (934, 960), (980, 994)]
Regular Bearish Divergences: [(513, 545), (644, 659), (702, 726)]
Hidden Bullish Divergences: [(417, 460), (653, 679), (691, 712), (712, 825), (897, 934), (1075, 1102), (1102, 1136)]
Hidden Bearish Divergences: [(101, 144), (144, 190), (361, 386), (386, 399), (485, 513), (659, 680), (962, 982), (1013, 1068), (1068, 1088)]


In [11]:
# Plot divergence lines on the chart
fig_div = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    row_heights=[0.7, 0.3],
    subplot_titles=("Candlestick Chart with Divergences", "Money Flow Index (MFI)")
)

# Add candlestick chart
fig_div.add_trace(
    go.Candlestick(
        x=df_plot['Date_str'],
        open=df_plot['Open'],
        high=df_plot['High'],
        low=df_plot['Low'],
        close=df_plot['Close'],
        name='OHLC',
        increasing_line_color='green',
        decreasing_line_color='red',
    ),
    row=1, col=1
)

# Add MFI
fig_div.add_trace(
    go.Scatter(
        x=df_plot['Date_str'],
        y=df_plot['mfi'],
        mode='lines',
        name='MFI',
        line=dict(color='blue', width=2)
      
    ),
    row=2, col=1
)

# Regular Bullish on PRICE
for p1, p2 in regular_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='green', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bullish<extra></extra>'
        ),
        row=1, col=1
    )

# Regular Bearish on PRICE
for p1, p2 in regular_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bearish<extra></extra>'
        ),
        row=1, col=1
    )

# Hidden Bullish on PRICE
for p1, p2 in hidden_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='lime', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Hidden Bullish<extra></extra>'
        ),
        row=1, col=1
    )

# Hidden Bearish on PRICE
for p1, p2 in hidden_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='orange', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Hidden Bearish<extra></extra>'
        ),
        row=1, col=1
    )

# Regular Bullish on MFI
for p1, p2 in regular_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='green', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bullish<extra></extra>'
        ),
        row=2, col=1
    )

# Regular Bearish on MFI
for p1, p2 in regular_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bearish<extra></extra>'
        ),
        row=2, col=1
    )

# Hidden Bullish on MFI
for p1, p2 in hidden_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='lime', dash='dot', width=2),
            showlegend=False,
            hovertemplate='Hidden Bullish<extra></extra>'
        ),
        row=2, col=1
    )

# Hidden Bearish on MFI
for p1, p2 in hidden_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='orange', dash='dot', width=2),
            showlegend=False,
            hovertemplate='Hidden Bearish<extra></extra>'
        ),
        row=2, col=1
    )

# Add overbought/oversold lines (faded)
fig_div.add_hline(y=70, line_dash="dash", line_color="rgba(128, 128, 128, 0.2)", row=2)
fig_div.add_hline(y=30, line_dash="dash", line_color="rgba(128, 128, 128, 0.2)", row=2)

# Add legend entries manually
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='green', dash='dash', width=2), 
                         name='Regular Bullish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='red', dash='dash', width=2), 
                         name='Regular Bearish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='lime', dash='dot', width=2), 
                         name='Hidden Bullish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='orange', dash='dot', width=2), 
                         name='Hidden Bearish'), row=1, col=1)

# Update layout
fig_div.update_layout(
    height=700,
    width=1400,
    title_text="Trading View with MFI Divergences",
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0, y=1)
)

fig_div.update_yaxes(title_text="Price", row=1, col=1)
fig_div.update_yaxes(title_text="MFI", row=2, col=1)

# Print summary
print(f"Regular Bullish Divergences: {len(regular_bullish)}")
print(f"Regular Bearish Divergences: {len(regular_bearish)}")
print(f"Hidden Bullish Divergences: {len(hidden_bullish)}")
print(f"Hidden Bearish Divergences: {len(hidden_bearish)}")

fig_div.show()

Regular Bullish Divergences: 7
Regular Bearish Divergences: 3
Hidden Bullish Divergences: 7
Hidden Bearish Divergences: 9


In [12]:
y_r_bullish = [y for _, y in regular_bullish]
y_r_bearish = [y for _, y in regular_bearish]
y_h_bullish = [y for _, y in hidden_bullish]
y_h_bearish = [y for _, y in hidden_bearish]

In [13]:
candle_intervals= [5,10,20,40,60]
results=[]

In [14]:
all_data=[]
for index in y_r_bullish:
    all_data.append((index, "Regular Bullish"))
for index in y_r_bearish:
    all_data.append((index, "Regular Bearish"))
for index in y_h_bullish:
    all_data.append((index, "Hidden Bullish"))
for index in y_h_bearish:
    all_data.append((index, "Hidden Bearish"))
    

In [15]:
symbol= df["Symbol"][0]
symbol

'FINANCE'

In [16]:
for i,pattern in all_data:
    if i+max(candle_intervals)>= len(df):
        continue
    for j in candle_intervals:
        price_now = df["Close"].iloc[i]
        price_future = df["Close"].iloc[i+j]

        pc= (price_future-price_now)/price_now *100

        results.append({
            "Sector": symbol,
            "Pattern":pattern,
            "Interval":j,
            "Start Index":i,
            "EndIndex": i+j,
            "Price Now":price_now,
            "Price After Interval":price_future,
            "Percentage Change": pc
        })
        

In [17]:
results = pd.DataFrame(results)

In [18]:
results

,Sector,Pattern,Interval,Start Index,EndIndex,Price Now,Price After Interval,Percentage Change
0,FINANCE,Regular Bullish,5,294,299,1305.48,1438.36,10.178632
1,FINANCE,Regular Bullish,10,294,304,1305.48,1547.48,18.537243
2,FINANCE,Regular Bullish,20,294,314,1305.48,1598.21,22.423170
3,FINANCE,Regular Bullish,40,294,334,1305.48,1788.34,36.987162
4,FINANCE,Regular Bullish,60,294,354,1305.48,1668.35,27.795906
...,...,...,...,...,...,...,...,...
115,FINANCE,Hidden Bearish,5,1088,1093,2345.95,2285.87,-2.561009
116,FINANCE,Hidden Bearish,10,1088,1098,2345.95,2303.73,-1.799697
117,FINANCE,Hidden Bearish,20,1088,1108,2345.95,2321.52,-1.041369
118,FINANCE,Hidden Bearish,40,1088,1128,2345.95,2535.82,8.093523


In [19]:
results.to_excel(f"{symbol}_MFI_Divergence.xlsx",index=False)